#### Homework 4

In [1]:
import numpy as np
import cv2

def run_main():
    cap = cv2.VideoCapture(0)
    cap.set(cv2.cv.CV_CAP_PROP_FRAME_WIDTH, 1280)
    cap.set(cv2.cv.CV_CAP_PROP_FRAME_HEIGHT, 720)

    while(True):
        ret, frame = cap.read()
        roi = frame[0:500, 0:500]
        gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)

        gray_blur = cv2.GaussianBlur(gray, (15, 15), 0)
        thresh = cv2.adaptiveThreshold(gray_blur, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                       cv2.THRESH_BINARY_INV, 11, 1)

        kernel = np.ones((3, 3), np.uint8)
        closing = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE,
        kernel, iterations=4)

        cont_img = closing.copy()
        contours, hierarchy = cv2.findContours(cont_img, cv2.RETR_EXTERNAL,
                                               cv2.CHAIN_APPROX_SIMPLE)

        for cnt in contours:
            area = cv2.contourArea(cnt)
            if area < 2000 or area > 4000:
                continue

            if len(cnt) < 5:
                continue

            ellipse = cv2.fitEllipse(cnt)
            cv2.ellipse(roi, ellipse, (0,255,0), 2)

        cv2.imshow("Morphological Closing", closing)
        cv2.imshow("Adaptive Thresholding", thresh)
        cv2.imshow('Contours', roi)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

if __name__ == "__main__":
    run_main()

AttributeError: module 'cv2' has no attribute 'cv'

In [10]:
import cv2
import numpy as np

def run_main():

    # ======================
    # 1. 이미지 읽기
    # ======================
    img = cv2.imread("sIMG_8253.JPG")

    if img is None:
        raise FileNotFoundError("이미지를 찾을 수 없습니다. 경로 확인하세요.")

    # ======================
    # 2. 크기 줄이기
    # ======================
    img = cv2.resize(img, None, fx=0.5, fy=0.5)

    # ======================
    # 3. grayscale + blur
    # ======================
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    gray_blur = cv2.GaussianBlur(gray, (15, 15), 0)

    # ======================
    # 4. threshold (thresh)
    # ======================
    thresh = cv2.adaptiveThreshold(
        gray_blur,
        255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY_INV,
        11,
        1
    )

    # ======================
    # 5. morphological closing
    # ======================
    kernel = np.ones((3, 3), np.uint8)

    closing = cv2.morphologyEx(
        thresh,
        cv2.MORPH_CLOSE,
        kernel,
        iterations=4
    )

    # ======================
    # 6. 동전 검출 (HoughCircles)
    # ======================
    circles = cv2.HoughCircles(
        gray_blur,
        cv2.HOUGH_GRADIENT,
        dp=1.2,
        minDist=50,
        param1=100,
        param2=30,
        minRadius=20,
        maxRadius=200
    )

    # ======================
    # 7. 원 그리기
    # ======================
    result = img.copy()

    if circles is not None:
        circles = np.uint16(np.around(circles))

        for i in circles[0]:
            center = (i[0], i[1])
            radius = i[2]

            cv2.circle(result, center, radius, (0, 255, 0), 3)
            cv2.circle(result, center, 2, (0, 0, 255), 3)

    # ======================
    # 8. 결과 출력
    # ======================
    cv2.imshow("Adaptive Threshold", thresh)
    cv2.imshow("Morphological Closing", closing)
    cv2.imshow("Coin Detection", result)

    cv2.waitKey(0)
    cv2.destroyAllWindows()


if __name__ == "__main__":
    run_main()